In [1]:
import json
import os
from google.colab import drive

drive.mount('/content/drive')
ocr_folder = "/content/drive/MyDrive/GOST_Project/ocr_results2"

label_studio_json = (
    "/content/drive/MyDrive/GOST_Project/"
    "project-2.json"
)

output_json = (
    "/content/drive/MyDrive/GOST_Project/"
    "layoutlm_dataset_2.json"
)


Mounted at /content/drive


In [ ]:
from collections import Counter

with open(label_studio_json, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print("Всего задач:", len(tasks))

annotated = 0

for task in tasks:

    anns = task.get("annotations", [])

    if len(anns) > 0:
        annotated += 1

print("Размечено:", annotated)

Всего задач: 330
Размечено: 330


In [ ]:
from collections import Counter

counter = Counter(ls_names)

duplicates = {
    k:v
    for k,v in counter.items()
    if v > 1
}

print("Дубликатов:", len(duplicates))

for k,v in list(duplicates.items())[:30]:
    print(v, k)

Дубликатов: 33
2 alekseeva_e_a_page_1.png
2 alekseeva_e_a_page_37.png
2 tohtueva_e_a_page_1.png
2 tohtueva_e_a_page_2.png
2 tohtueva_e_a_page_24.png
2 tohtueva_e_a_page_25.png
2 tohtueva_e_a_page_26.png
2 tohtueva_e_a_page_27.png
2 trubacheva_v_p_page_1.png
2 Вихорев_Текст_page_1.png
2 Вихорев_Текст_page_2.png
2 Вихорев_Текст_page_25.png
2 Вихорев_Текст_page_26.png
2 Вихорев_Текст_page_27.png
2 Вихорев_Текст_page_3.png
2 Воробьев_ТекстВКР_page_1.png
2 Воробьев_ТекстВКР_page_39.png
2 Гащенко_диплом_page_1.png
2 Гащенко_диплом_page_20.png
2 Гащенко_диплом_page_21.png
2 Гащенко_диплом_page_22.png
2 Гащенко_диплом_page_23.png
2 Гащенко_диплом_page_24.png
2 Гащенко_диплом_page_27.png
2 Гащенко_диплом_page_28.png
2 Гащенко_диплом_page_34.png
2 Гащенко_диплом_page_35.png
2 Гащенко_диплом_page_36.png
2 Гащенко_диплом_page_37.png
2 Гащенко_диплом_page_56.png


In [ ]:
print(len(ls_names))
print(len(set(ls_names)))

330
297


In [ ]:
from collections import Counter

counter = Counter(ls_names)

print("Всего уникальных файлов:", len(counter))

print("Файлов с дублями:", sum(v > 1 for v in counter.values()))

print("Лишних записей:", sum(v - 1 for v in counter.values()))

Всего уникальных файлов: 297
Файлов с дублями: 33
Лишних записей: 33


In [ ]:
import json

with open(label_studio_json, "r", encoding="utf-8") as f:
    tasks = json.load(f)

task = tasks[0]

print(task.keys())

dict_keys(['id', 'annotations', 'file_upload', 'drafts', 'predictions', 'data', 'meta', 'created_at', 'updated_at', 'allow_skip', 'inner_id', 'total_annotations', 'cancelled_annotations', 'total_predictions', 'comment_count', 'unresolved_comment_count', 'last_comment_updated_at', 'project', 'updated_by', 'comment_authors'])


In [ ]:
import json

with open(label_studio_json, "r", encoding="utf-8") as f:
    tasks = json.load(f)

task = tasks[0]

annotation = task["annotations"][0]

types = {}

for item in annotation["result"]:
    t = item["type"]
    types[t] = types.get(t, 0) + 1

print(types)

{'rectangle': 5, 'labels': 5}


In [ ]:
import unicodedata
from pathlib import Path
from urllib.parse import unquote


print("Загружаем OCR.")

ocr_map = {}

for file in os.listdir(ocr_folder):

    if not file.endswith(".json"):
        continue

    path = os.path.join(ocr_folder, file)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_name = unicodedata.normalize(
        "NFC",
        data["image"]
    )

    ocr_map[image_name] = data

print("OCR файлов:", len(ocr_map))

# LABEL STUDIO

print("Загружаем Label Studio...")

with open(label_studio_json, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print("Задач:", len(tasks))

dataset = []

# ОСНОВНОЙ ЦИКЛ

for task_idx, task in enumerate(tasks):

    raw_image = task["data"]["image"]

    filename = Path(
        unquote(raw_image)
    ).name


    if "-" in filename:

        left, right = filename.split("-", 1)

        if len(left) == 8:
            filename = right

    filename = unicodedata.normalize(
        "NFC",
        filename
    )

    # OCR

    if "words" in task["data"]:

        words = task["data"]["words"]

    else:

        if filename not in ocr_map:

            print("Нет OCR:", filename)
            continue

        words = ocr_map[filename]["words"]

    # РАЗМЕТКА

    annotation = task["annotations"][0]

    results = annotation["result"]

    rectangles = {}
    labels = {}

    for r in results:

        rid = r["id"]

        if r["type"] == "rectangle":

            rectangles[rid] = r

        elif r["type"] == "labels":

            labels[rid] = r

    regions = []

    for rid in rectangles:

        if rid not in labels:
            continue

        rect = rectangles[rid]
        lab = labels[rid]

        rect_val = rect["value"]

        x = rect_val["x"]
        y = rect_val["y"]
        w = rect_val["width"]
        h = rect_val["height"]

        label = lab["value"]["labels"][0]

        img_w = rect["original_width"]
        img_h = rect["original_height"]

        x1 = x * img_w / 100
        y1 = y * img_h / 100
        x2 = (x + w) * img_w / 100
        y2 = (y + h) * img_h / 100

        regions.append({
            "label": label,
            "bbox": [x1, y1, x2, y2]
        })

    # СЛОВА

    page_words = []

    for word in words:

        bbox = word["bbox"]

        x_center = (bbox[0] + bbox[2]) / 2
        y_center = (bbox[1] + bbox[3]) / 2

        assigned_label = "OTHER"

        for region in regions:

            x1, y1, x2, y2 = region["bbox"]

            if (
                x1 <= x_center <= x2
                and
                y1 <= y_center <= y2
            ):
                assigned_label = region["label"]
                break

        page_words.append({
            "text": word["text"],
            "bbox": bbox,
            "label": assigned_label
        })

    dataset.append({
        "image": filename,
        "words": page_words
    })

    if (task_idx + 1) % 25 == 0:
        print(
            f"Обработано {task_idx + 1} страниц"
        )

with open(
    output_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dataset,
        f,
        ensure_ascii=False
    )

print("ГОТОВО")
print("Страниц:", len(dataset))
print("Файл:", output_json)

# СТАТИСТИКА

stats = {}

for page in dataset:

    for w in page["words"]:

        lbl = w["label"]

        stats[lbl] = stats.get(lbl, 0) + 1

print("\nСтатистика классов:")

for k, v in sorted(
    stats.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{k}: {v}")

Загружаем OCR...
OCR файлов: 372
Загружаем Label Studio...
Задач: 330
Обработано 25 страниц
Обработано 50 страниц
Обработано 75 страниц
Обработано 100 страниц
Обработано 125 страниц
Обработано 150 страниц
Обработано 175 страниц
Обработано 200 страниц
Обработано 225 страниц
Обработано 250 страниц
Обработано 275 страниц
Обработано 300 страниц
Обработано 325 страниц
Нет OCR: Жемчужина_диплом_1_page_1.png
Нет OCR: Жемчужина_диплом_1_page_2.png
Нет OCR: Жемчужина_диплом_1_page_30.png

ГОТОВО
Страниц: 327
Файл: /content/drive/MyDrive/GOST_Project/layoutlm_dataset_2.json

Статистика классов:
BODY: 12374
REFERENCE: 5371
FIGURE: 1253
TOC: 1200
TABLE: 1067
FOOTNOTE: 644
FORMULA: 443
FIGURE_CAPTION: 329
SECTION_HEADER: 292
OTHER: 257
TITLE: 244
TABLE_CAPTION: 85


In [ ]:
ocr_data["words"][0]

{'text': 'Список литературы',
 'bbox': [233, 149, 739, 209],
 'confidence': 0.963402296430255}

In [ ]:
len(ocr_data["words"])

152

In [ ]:
import json

ocr_file = "/content/drive/MyDrive/GOST_Project/ocr_results/Вихорев_Текст_page_17.json"

with open(ocr_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print(data.keys())

print("\nПервый элемент words:")
print(data["words"][0])

dict_keys(['image', 'words'])

Первый элемент words:
{'text': '3.3', 'bbox': [271, 159, 345, 203], 'confidence': 0.9990194439888}


In [ ]:
print(data["words"][0])

{'text': '3.3', 'bbox': [271, 159, 345, 203], 'confidence': 0.9990194439888}


In [ ]:
print(data["words"][1])
print(data["words"][2])

{'text': 'Процедура деления неразмеченной шутки', 'bbox': [383, 155, 1399, 222], 'confidence': 0.86753498005837}
{'text': 'на', 'bbox': [390, 260, 450, 292], 'confidence': 0.9300382402608592}


In [ ]:
import json

with open(
    "/content/drive/MyDrive/GOST_Project/layoutlm_dataset_2.json",
    "r",
    encoding="utf-8"
) as f:
    dataset = json.load(f)

print("Страниц:", len(dataset))

for page in dataset[:3]:
    print()
    print(page["image"])
    print("Элементов:", len(page["words"]))

Страниц: 327

alekseeva_e_a_page_1.png
Элементов: 17

alekseeva_e_a_page_10.png
Элементов: 121

alekseeva_e_a_page_2.png
Элементов: 15


In [ ]:
from PIL import Image
import os

images_folder = "/content/drive/MyDrive/GOST_Project/общие_фото_новинка"

sizes = []

for f in os.listdir(images_folder):
    if f.endswith(".png"):
        img = Image.open(os.path.join(images_folder, f))
        sizes.append(img.size)

print("Уникальные размеры:")

for s in sorted(set(sizes)):
    print(s)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Уникальные размеры:
(1654, 2338)
(1654, 2339)
(1656, 2339)
(1700, 2200)
